In [1]:
# Импортируем алхимию и создаем соединение
from sqlalchemy import create_engine, text
import os
from dotenv import load_dotenv
import pandas as pd
import numpy as np

load_dotenv()

POSTGRES_USER = os.getenv('POSTGRES_USER')
POSTGRES_PASSWORD = os.getenv('POSTGRES_PASSWORD')
POSTGRES_DB = os.getenv('POSTGRES_DB')
POSTGRES_HOST= os.getenv('POSTGRES_HOST', 'localhost')

conn = create_engine(f'postgresql://{POSTGRES_USER}:{POSTGRES_PASSWORD}@{POSTGRES_HOST}:5432/{POSTGRES_DB}')


In [4]:
sql = '''
    SELECT 
        *
    FROM raw.extract_api_jobs
'''

df = pd.read_sql(sql, conn)
df

,id,endpoint,request_payload,response_data,status_code,loaded_at
0,1,MarketDataService/GetCandles,"{'to': '2026-06-30T06:08:41.455247+00:00', 'fi...","{'candles': [{'low': {'nano': 230000000, 'unit...",200,2026-06-30 06:08:41.585007+00:00
1,2,MarketDataService/GetCandles,"{'to': '2026-06-30T06:39:42.378524+00:00', 'fi...","{'candles': [{'low': {'nano': 0, 'units': '297...",200,2026-06-30 06:39:42.553008+00:00
2,3,MarketDataService/GetCandles,"{'to': '2026-07-01T05:15:45.774673+00:00', 'fi...","{'candles': [{'low': {'nano': 890000000, 'unit...",200,2026-07-01 05:15:45.932152+00:00
3,4,MarketDataService/GetCandles,"{'to': '2026-07-02T05:15:47.038426+00:00', 'fi...","{'candles': [{'low': {'nano': 850000000, 'unit...",200,2026-07-02 05:15:51.891103+00:00
4,5,MarketDataService/GetCandles,"{'to': '2026-07-02T05:15:49.436653+00:00', 'fi...","{'candles': [{'low': {'nano': 850000000, 'unit...",200,2026-07-02 05:15:54.170052+00:00
5,6,MarketDataService/GetCandles,"{'to': '2026-07-03T05:46:40.445665+00:00', 'fi...","{'candles': [{'low': {'nano': 670000000, 'unit...",200,2026-07-03 05:46:44.184541+00:00
6,7,MarketDataService/GetCandles,"{'to': '2026-07-03T06:38:12.709434+00:00', 'fi...","{'candles': [{'low': {'nano': 610000000, 'unit...",200,2026-07-03 06:38:15.254333+00:00
7,8,MarketDataService/GetCandles,"{'to': '2026-07-03T06:38:30.390869+00:00', 'fi...","{'candles': [{'low': {'nano': 610000000, 'unit...",200,2026-07-03 06:38:32.991671+00:00
8,9,InstrumentsService/GetAssets,"{'instrumentType': 'INSTRUMENT_TYPE_SHARE', 'i...",{'assets': [{'uid': 'b6a73950-20a8-46c7-8b49-9...,200,2026-07-03 06:38:53.395885+00:00
9,10,InstrumentsService/GetAssets,"{'instrumentType': 'INSTRUMENT_TYPE_SHARE', 'i...",{'assets': [{'uid': 'b6a73950-20a8-46c7-8b49-9...,200,2026-07-03 06:38:55.969439+00:00


In [20]:
sql = '''
drop table if exists backfill_candles;
create table backfill_candles as
with success_jobs as (
	select
		id,
		jsonb_array_length(response_data -> 'candles') AS resp_candles_cnt,
		(request_payload #>> '{from}')::date as req_from_dt,
		(request_payload #>> '{to}')::date as req_to_dt,
		request_payload #>> '{figi}' as req_figi,
		request_payload #>> '{interval}' as req_interval,
		status_code,
		loaded_at as db_loaded_dttm
	from raw.extract_api_jobs
	where endpoint = 'MarketDataService/GetCandles'
	and status_code = 200
),
unique_par as (
	select distinct
		req_figi,
        req_interval
    from success_jobs
)

select distinct
	c.date_id as from_dt,
    up.req_figi as figi,
    up.req_interval as interval
from dim_calendar c
cross join unique_par up
where 1=1
and c.date_id <= current_date
and c.date_id >= '2026-06-01'

except -- возращает строки из первого запроса, которые отсутствуют во втором

select distinct
	req_from_dt as from_dt,
    req_figi as figi,
    req_interval as interval
from success_jobs

order by 1
'''

with conn.begin() as ex:
    ex.execute(text(sql))

In [21]:
sql = '''
select *
from raw.candles_interval_limits
'''

df = pd.read_sql(sql, conn)
df

,interval_full,interval_short,max_limit,candles_per_day,max_days_calc,description,created_at
0,CANDLE_INTERVAL_1_MIN,1min,2400,1440.000000,1,От 1 минуты до 1 дня,2026-07-06 06:38:47.649481+00:00
1,CANDLE_INTERVAL_2_MIN,2min,1200,720.000000,1,От 2 минут до 1 дня,2026-07-06 06:38:47.649481+00:00
2,CANDLE_INTERVAL_3_MIN,3min,750,480.000000,1,От 3 минут до 1 дня,2026-07-06 06:38:47.649481+00:00
3,CANDLE_INTERVAL_5_MIN,5min,2400,288.000000,8,От 5 минут до недели,2026-07-06 06:38:47.649481+00:00
4,CANDLE_INTERVAL_10_MIN,10min,1200,144.000000,8,От 10 минут до недели,2026-07-06 06:38:47.649481+00:00
5,CANDLE_INTERVAL_15_MIN,15min,2400,96.000000,25,От 15 минут до 3 недель,2026-07-06 06:38:47.649481+00:00
6,CANDLE_INTERVAL_30_MIN,30min,1200,48.000000,25,От 30 минут до 3 недель,2026-07-06 06:38:47.649481+00:00
7,CANDLE_INTERVAL_HOUR,hour,2400,24.000000,100,От 1 часа до 3 месяцев,2026-07-06 06:38:47.649481+00:00
8,CANDLE_INTERVAL_2_HOUR,2hour,2400,12.000000,200,От 2 часов до 3 месяцев,2026-07-06 06:38:47.649481+00:00
9,CANDLE_INTERVAL_4_HOUR,4hour,700,6.000000,116,От 4 часов до 3 месяцев,2026-07-06 06:38:47.649481+00:00


In [ ]:
sql = '''
SELECT *



FROM backfill_candles_blocks
'''

df = pd.read_sql(sql, conn)
df

,figi,interval,interval_short,from_dt,to_dt,days_count,max_days_calc
0,BBG004730N88,CANDLE_INTERVAL_15_MIN,15min,2026-06-01,2026-06-25,25,25
1,BBG004730N88,CANDLE_INTERVAL_15_MIN,15min,2026-06-26,2026-07-07,5,25


In [ ]:
sql = '''
    select
        id,
        jsonb_array_length(response_data -> 'candles') AS resp_candles_cnt,
        (request_payload #>> '{from}') as req_from_dt,
        (request_payload #>> '{to}')::date as req_to_dt,
        request_payload #>> '{figi}' as req_figi,
        request_payload #>> '{interval}' as req_interval,
        status_code,
        loaded_at as db_loaded_dttm
    from raw.extract_api_jobs
    where endpoint = 'MarketDataService/GetCandles'
    and status_code = 200
'''

df = pd.read_sql(sql, conn)
df

,id,resp_candles_cnt,req_from_dt,req_to_dt,req_figi,req_interval,status_code,db_loaded_dttm
0,1,70,2026-06-29,2026-06-30,BBG004730N88,CANDLE_INTERVAL_15_MIN,200,2026-06-30 06:08:41.585007+00:00
1,2,70,2026-06-29,2026-06-30,BBG004730N88,CANDLE_INTERVAL_15_MIN,200,2026-06-30 06:39:42.553008+00:00
2,3,69,2026-06-30,2026-07-01,BBG004730N88,CANDLE_INTERVAL_15_MIN,200,2026-07-01 05:15:45.932152+00:00
3,4,69,2026-07-01,2026-07-02,BBG004730N88,CANDLE_INTERVAL_15_MIN,200,2026-07-02 05:15:51.891103+00:00
4,5,69,2026-07-01,2026-07-02,BBG004730N88,CANDLE_INTERVAL_15_MIN,200,2026-07-02 05:15:54.170052+00:00
5,6,70,2026-07-02,2026-07-03,BBG004730N88,CANDLE_INTERVAL_15_MIN,200,2026-07-03 05:46:44.184541+00:00
6,7,70,2026-07-02,2026-07-03,BBG004730N88,CANDLE_INTERVAL_15_MIN,200,2026-07-03 06:38:15.254333+00:00
7,8,70,2026-07-02,2026-07-03,BBG004730N88,CANDLE_INTERVAL_15_MIN,200,2026-07-03 06:38:32.991671+00:00
8,12,88,2026-07-03,2026-07-04,BBG004730N88,CANDLE_INTERVAL_15_MIN,200,2026-07-04 08:30:58.814825+00:00
9,13,89,2026-07-03,2026-07-04,BBG004730N88,CANDLE_INTERVAL_15_MIN,200,2026-07-04 08:32:18.213452+00:00
